# Day 7: Capstone (solution)

Reference solutions. Every cell is runnable; `practice.ipynb` is the blank-page twin.

## `s1_data.py`: Data — Counter algebra + set overlap (Day 1 revisited).

In [ ]:
from collections import Counter
a, b = Counter("the cat sat".split()), Counter("the dog sat".split())

In [ ]:
from collections import Counter

assert set(a) & set(b) == {"the", "sat"}
assert len(set(a) & set(b)) / len(set(a) | set(b)) == 0.5    # Jaccard
assert (a & b) == Counter({"the": 1, "sat": 1}) and (a + b)["the"] == 2

**Then, vary it:**

- VARIATION: compare two real paragraphs; which words are unique to each?

## `s2_flow.py`: Flow — lazy tokenizer generator.

In [ ]:
import re
def tokens(text):
    for m in re.finditer(r"[A-Za-z']+|\d+|[^\s\w]", text):
        t = m.group()
        if t.isdigit(): yield ("NUM", t)
        elif t[0].isalpha(): yield ("WORD", t.lower())
        else: yield ("PUNCT", t)

In [ ]:
assert list(tokens("Hi 5!")) == [("WORD", "hi"), ("NUM", "5"), ("PUNCT", "!")]

**Then, vary it:**

- VARIATION: rewrite the if/elif chain with match + guards

## `s3_functions.py`: Functions — cache + timing decorator.

In [ ]:
import functools, re, time
from functools import cache
@cache
def normalize(t: str) -> tuple[str, ...]:
    return tuple(re.findall(r"[a-z']+", t.lower()))

In [ ]:
assert normalize("A b A") == ("a", "b", "a")

In [ ]:
normalize("A b A")

In [ ]:
assert normalize.cache_info().hits == 1

In [ ]:
def timed(fn):
    @functools.wraps(fn)
    def w(*a, **k):
        t = time.perf_counter()
        r = fn(*a, **k)
        w.elapsed = time.perf_counter() - t
        return r
    return w
@timed
def f(): return 1

In [ ]:
assert f() == 1 and f.elapsed >= 0

**Then, vary it:**

- VARIATION: why is returning a tuple (not list) important for @cache users?

## `s4_classes.py`: Classes — frozen+slots dataclass, Protocol strategy.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from typing import Protocol
@dataclass(frozen=True, slots=True)
class TextStats:
    total: int
    unique: int
    top: tuple[tuple[str, int], ...]
    @classmethod
    def of(cls, words, n=2):
        c = Counter(words)
        return cls(c.total(), len(c), tuple(c.most_common(n)))
class Scorer(Protocol):
    def score(self, s: TextStats) -> float: ...
class Diversity:
    def score(self, s): return s.unique / s.total
s = TextStats.of("a b a c".split())

In [ ]:
assert (s.total, s.unique, s.top[0]) == (4, 3, ("a", 2)) and Diversity().score(s) == 0.75

**Then, vary it:**

- VARIATION: add a second Scorer (e.g. average word length) and pick via a dict

## `s5_errors.py`: Errors — streaming file count + domain error.

In [ ]:
import tempfile
from pathlib import Path
class EmptyTextError(ValueError): pass
def count_file(path):
    n = 0
    with Path(path).open() as f:
        for line in f: n += len(line.split())
    if n == 0: raise EmptyTextError(path)
    return n

In [ ]:
from pathlib import Path
import tempfile

with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "a.txt"
    p.write_text("a b\nc\n")
    assert count_file(p) == 3
    p.write_text("\n")
    try: count_file(p)
    except EmptyTextError: pass
    else: raise AssertionError

**Then, vary it:**

- VARIATION: accept a directory and count all *.txt recursively

## `s6_stdlib.py`: Stdlib — bigrams with pairwise + Counter.

In [ ]:
import re
from collections import Counter
from itertools import pairwise
words = re.findall(r"[a-z']+", "the cat. the cat sat".lower())
bi = Counter(pairwise(words))

In [ ]:
assert bi[("the", "cat")] == 2 and bi.most_common(1) == [(("the", "cat"), 2)]

**Then, vary it:**

- VARIATION: trigrams with zip(words, words[1:], words[2:]); compare to pairwise

## `s7_concurrency.py`: Concurrency — same job via threads and asyncio.

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
docs = ["a b", "c", "d e f"]
def n(d): return len(d.split())

In [ ]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor() as ex:
    assert list(ex.map(n, docs)) == [2, 1, 3]

In [ ]:
async def main(): return await asyncio.gather(*(asyncio.to_thread(n, d) for d in docs))

In [ ]:
import asyncio

assert await main() == [2, 1, 3]

**Then, vary it:**

- VARIATION: write the ProcessPoolExecutor version (remember the __main__ guard)

## `s8_quality.py`: Quality — typed + doctest + benchmark, together.

In [ ]:
import doctest, re, timeit
from collections import Counter
def vocab_size(text: str) -> int:
    """
    >>> vocab_size("A a b")
    2
    >>> vocab_size("")
    0
    """
    return len(Counter(re.findall(r"[a-z']+", text.lower())))
assert doctest.testmod().failed == 0

In [ ]:
import doctest, re, timeit

assert timeit.timeit(lambda: vocab_size("a b c " * 200), number=50) < 5

**Then, vary it:**

- VARIATION: add a property-style check: vocab_size(t) <= len(t.split()) for random t

## `kata.py`: KATA — Day 1's top_words, naive vs idiomatic, proven equivalent + timed.

In [ ]:
import doctest, random, re, timeit
from collections import Counter
def top_naive(text, n=3):       # Day 1 version
    counts = {}
    for raw in text.lower().split():
        w = "".join(c for c in raw if c.isalnum())
        if w: counts[w] = counts.get(w, 0) + 1
    return sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:n]
def top_idiomatic(text: str, n: int = 3) -> list[tuple[str, int]]:
    """
    >>> top_idiomatic("The cat. The hat! A cat?", 2)
    [('cat', 2), ('the', 2)]
    """
    c = Counter(re.findall(r"[a-z0-9]+", text.lower()))
    return sorted(c.items(), key=lambda kv: (-kv[1], kv[0]))[:n]
random.seed(1)
vocab = ["the", "cat", "hat", "a", "dog"]
text = " ".join(random.choice(vocab) + random.choice(["", ",", "."]) for _ in range(5000))

In [ ]:
assert top_naive(text) == top_idiomatic(text)

In [ ]:
assert doctest.testmod().failed == 0
print("naive     ", timeit.timeit(lambda: top_naive(text), number=20))
print("idiomatic ", timeit.timeit(lambda: top_idiomatic(text), number=20))

**Then, vary it:**

- KATA: write the Day 1 solution from memory FIRST, then the idiomatic one, then compare.
Note the two differ on "don't" (naive -> dont, regex -> don, t): equal only on this input.